# Aula 18 - Grafos de Logística de Insumos e Produtos Acabados

Este notebook complementa a malha de processo das aulas anteriores com duas redes do layout de `Grafo_Logitica.jpeg`: um dígrafo de fluxo de materiais e outro para circulação interna de veículos. Os pesos são estimativas didáticas em metros, não medições da imagem.


In [ ]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple


@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str


class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError("Origem e destino devem pertencer ao grafo.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho


def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)


# GM: fluxo de materiais. Os pesos são distâncias internas estimadas para fins didáticos.
fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Portaria", "Balança de entrada", 25, "liberação e pesagem do recebimento"),
    ("Balança de entrada", "Recebimento / Galpão A", 55, "direcionamento à descarga"),
    ("Recebimento / Galpão A", "Box: ureia", 18, "armazenagem de fonte nitrogenada"),
    ("Recebimento / Galpão A", "Box: fosfato", 20, "armazenagem de fonte fosfatada"),
    ("Recebimento / Galpão A", "Box: KCl", 22, "armazenagem de cloreto de potássio"),
    ("Recebimento / Galpão A", "Box: orgânicos e micronutrientes", 25, "armazenagem de complementos"),
    ("Recebimento / Galpão A", "Tanques líquidos", 28, "recebimento de aditivos líquidos"),
    ("Box: ureia", "Moega de recepção", 35, "transferência por correia"),
    ("Box: fosfato", "Moega de recepção", 32, "transferência por correia"),
    ("Box: KCl", "Moega de recepção", 30, "transferência por correia"),
    ("Box: orgânicos e micronutrientes", "Moega de recepção", 38, "transferência por correia"),
    ("Moega de recepção", "Silos de dosagem", 40, "alimentação dos dosadores"),
    ("Tanques líquidos", "Silos de dosagem", 45, "dosagem de aditivos"),
    ("Silos de dosagem", "Moinho / misturador", 18, "homogeneização da fórmula"),
    ("Moinho / misturador", "Granulação", 20, "formação dos grânulos"),
    ("Granulação", "Secador / classificação", 32, "secagem e peneiramento"),
    ("Secador / classificação", "Ensacamento / paletização", 48, "acabamento e embalagem"),
    ("Ensacamento / paletização", "Estoque NPK 04-14-08", 35, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque NPK 10-10-10", 42, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque Organomineral Premium", 50, "endereçamento do produto acabado"),
    ("Estoque NPK 04-14-08", "Docas de expedição", 28, "separação para carregamento"),
    ("Estoque NPK 10-10-10", "Docas de expedição", 25, "separação para carregamento"),
    ("Estoque Organomineral Premium", "Docas de expedição", 32, "separação para carregamento"),
])

distancia, caminho = fluxo_materiais.menor_caminho("Portaria", "Docas de expedição")
print("Fluxo de materiais até a expedição")
print("  " + " -> ".join(caminho))
print(f"  Distância didática total: {distancia:.0f} m")
assert "Silos de dosagem" in caminho
assert "Ensacamento / paletização" in caminho

print("\nRotas dos insumos até a expedição")
for insumo in [
    "Box: ureia",
    "Box: fosfato",
    "Box: KCl",
    "Box: orgânicos e micronutrientes",
    "Tanques líquidos",
]:
    distancia_insumo, caminho_insumo = fluxo_materiais.menor_caminho(insumo, "Docas de expedição")
    print(f"  {insumo}: {distancia_insumo:.0f} m ({' -> '.join(caminho_insumo)})")


# GV: circulação interna. Cada direção permitida é uma aresta explícita.
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Portaria", "Balança de entrada", 25, "entrada de veículos"),
    ("Balança de entrada", "Pátio de recebimento", 55, "acesso à descarga"),
    ("Pátio de recebimento", "Galpão A", 40, "rota de descarga"),
    ("Galpão A", "Área de processamento", 65, "rota interna de serviço"),
    ("Área de processamento", "Galpão B", 75, "rota para armazenagem"),
    ("Galpão B", "Docas de expedição", 30, "rota de carregamento"),
    ("Docas de expedição", "Balança de saída", 60, "saída de veículos"),
    ("Balança de saída", "Portaria", 35, "liberação de saída"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

distancia_saida, rota_saida = circulacao_veiculos.menor_caminho(
    "Docas de expedição", "Portaria"
)
print("\nCirculação de veículo após o carregamento")
print("  " + " -> ".join(rota_saida))
print(f"  Distância didática total: {distancia_saida:.0f} m")
assert rota_saida == ["Docas de expedição", "Balança de saída", "Portaria"]
